<a href="https://colab.research.google.com/github/miriamamin1213-ux/AUC_Curves/blob/main/HancockGPTL4BSHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# HANCOCK GPT-2 Last 4 Block Unfrozen
# ============================================================
# 1. IMPORT LIBRARIES, SET SEEDS AND SELECT DEVICE
# ============================================================
import os, random, json, gdown
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

SEED = 42

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. LOAD AND MERGE HANCOCK CLINICAL/PATHOLOGICAL DATA
# ============================================================
CLINICAL_FILE = "clinical_data.json"
PATHOLOGICAL_FILE = "pathological_data.json"

if not os.path.exists(CLINICAL_FILE):
    if gdown.download(
        id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
        output=CLINICAL_FILE, quiet=False
    ) is None:
        raise FileNotFoundError("Could not download clinical_data.json.")

if not os.path.exists(PATHOLOGICAL_FILE):
    if gdown.download(
        id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
        output=PATHOLOGICAL_FILE, quiet=False
    ) is None:
        raise FileNotFoundError("Could not download pathological_data.json.")

def load_json_table(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        for key in ["data", "records", "patients", "items"]:
            if key in obj and isinstance(obj[key], list):
                return pd.DataFrame(obj[key])
        try:
            return pd.DataFrame(obj)
        except ValueError:
            pass
    raise ValueError(f"Could not convert {path} to a table.")

clinical_df = load_json_table(CLINICAL_FILE)
pathological_df = load_json_table(PATHOLOGICAL_FILE)
if ("patient_id" not in clinical_df.columns or
        "patient_id" not in pathological_df.columns):
    raise KeyError("Both JSON files must contain 'patient_id'.")

df = clinical_df.merge(
    pathological_df, on="patient_id", how="left",
    suffixes=("", "_path")
)
print("Clinical shape:", clinical_df.shape)
print("Pathology shape:", pathological_df.shape)
print("Merged shape:", df.shape)

# ============================================================
# 3. SELECT SIX FEATURES AND DEFINE THE HPV TARGET
# ============================================================
def resolve_col(candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"Required column not found: {candidates}. "
            f"Available columns: {df.columns.tolist()}"
        )
    return None

target_col = resolve_col(
    ["hpv_association_p16", "hpv_association_p16_path"],
    required=True
)
feature_candidates = {
    "age_at_initial_diagnosis": [
        "age_at_initial_diagnosis", "age_at_initial_diagnosis_path"
    ],
    "sex": ["sex", "sex_path"],
    "smoking_status": ["smoking_status", "smoking_status_path"],
    "primary_tumor_site": [
        "primary_tumor_site", "primary_tumor_site_path"
    ],
    "primarily_metastasis": [
        "primarily_metastasis", "primarily_metastasis_path"
    ],
    "pN_stage": [
        "pN_stage", "pN_stage_path",
        "pathological_N_stage", "pathological_n_stage"
    ]
}
feature_map = {
    name: resolve_col(cols, required=True)
    for name, cols in feature_candidates.items()
}
features = list(feature_candidates.keys())
df = df.dropna(subset=[target_col]).copy()

def normalize_target(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["negative", "neg", "0", "false", "no"]:
        return 0
    if s in ["positive", "pos", "1", "true", "yes"]:
        return 1
    try:
        n = float(s)
        if n in [0, 1]:
            return int(n)
    except ValueError:
        pass
    return np.nan

df["HPV_Label"] = df[target_col].apply(normalize_target)
df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
feature_df = df[[feature_map[c] for c in features]].copy()
feature_df = feature_df.rename(
    columns={feature_map[c]: c for c in features}
)
df = pd.concat([feature_df, df[["HPV_Label"]]], axis=1)
df = df.loc[:, ~df.columns.duplicated()].copy()
df["age_at_initial_diagnosis"] = pd.to_numeric(
    df["age_at_initial_diagnosis"], errors="coerce"
)
print("Patients:", len(df))
print("HPV classes (0=negative, 1=positive):")
print(df["HPV_Label"].value_counts().sort_index())
print("Features:", features)
print("N-stage source:", feature_map["pN_stage"])

# ============================================================
# 4. STRATIFIED TRAIN/VALIDATION/TEST SPLIT (65/15/20)
# ============================================================
train_val_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED,
    stratify=df["HPV_Label"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.1875, random_state=SEED,
    stratify=train_val_df["HPV_Label"]
)
train_df, val_df, test_df = (
    split.copy() for split in [train_df, val_df, test_df]
)

# ============================================================
# 5. IMPUTE MISSING VALUES USING TRAINING DATA ONLY
# ============================================================
fill_values = {}
for c in features:
    if pd.api.types.is_numeric_dtype(train_df[c]):
        value = train_df[c].median()
        if pd.isna(value):
            value = 0
    else:
        modes = train_df[c].dropna().astype(str).mode()
        value = modes.iloc[0] if not modes.empty else "Unknown"
    fill_values[c] = value
    for split in [train_df, val_df, test_df]:
        split[c] = split[c].fillna(value)
        if split[c].dtype == object:
            split[c] = split[c].astype(str).replace({
                "": "Unknown", "nan": "Unknown", "None": "Unknown"
            })

y_train = train_df["HPV_Label"].to_numpy(dtype=int)
y_val = val_df["HPV_Label"].to_numpy(dtype=int)
y_test = test_df["HPV_Label"].to_numpy(dtype=int)
print("Train:", len(train_df), np.bincount(y_train, minlength=2))
print("Validation:", len(val_df), np.bincount(y_val, minlength=2))
print("Test:", len(test_df), np.bincount(y_test, minlength=2))

# ============================================================
# 6. CONVERT EXACTLY SIX CLINICAL FEATURES INTO TEXT
# ============================================================

def display_value(x):
    if pd.isna(x):
        return "Unknown"

    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        return str(int(x))

    return str(x).strip()


def patient_to_text(row):

    # Feature 1: Age
    age = display_value(row["age_at_initial_diagnosis"])

    # Feature 2: Sex
    sex_value = str(row["sex"]).strip().lower()

    if sex_value in ["1", "male", "m"]:
        sex = "male"
    elif sex_value in ["0", "female", "f"]:
        sex = "female"
    else:
        sex = display_value(row["sex"])

    # Feature 3: Smoking status
    smoking_value = str(row["smoking_status"]).strip().lower()

    if smoking_value in ["0", "non-smoker", "nonsmoker", "never"]:
        smoking = "a non-smoker"
    elif smoking_value in ["1", "smoker", "current smoker", "yes"]:
        smoking = "a smoker"
    else:
        smoking = display_value(row["smoking_status"])

    # Feature 4: Primary tumour site
    site = display_value(row["primary_tumor_site"])

    # Feature 5: Primarily metastasis
    metastasis = display_value(row["primarily_metastasis"])

    # Feature 6: Pathological N stage
    n_stage = display_value(row["pN_stage"])

    prompt = (
        "Q: Predict this patient's Human Papillomavirus (HPV) status "
        "using the clinical information available. "

        f"This patient is {age} years old and is {sex}. "
        f"The patient is {smoking}. "
        f"The primary tumour site is {site}. "
        f"The primarily metastasis status is {metastasis}. "
        f"Pathological nodal stage is {n_stage}. "

        "Based on these six clinical features, "
        "predict whether the patient is HPV positive or HPV negative. "
        "Answer with only Yes or No.\nA:"
    )

    return prompt


# Verify exactly six input features
features = [
    "age_at_initial_diagnosis",
    "sex",
    "smoking_status",
    "primary_tumor_site",
    "primarily_metastasis",
    "pN_stage"
]

assert len(features) == 6

print("Number of clinical features:", len(features))
print("Features:", features)

print("\nExample clinical prompt:")
print(patient_to_text(df.iloc[0]))
# ============================================================
# 7. LOAD TOKENIZER AND SELECT MAXIMUM LENGTH
# ============================================================
MODEL_NAME = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Determine the length policy from training prompts only to avoid
# using validation/test data to choose preprocessing parameters.

# Generate clinical text prompts for each dataset split
train_texts = [
    patient_to_text(row)
    for _, row in train_df.iterrows()
]

val_texts = [
    patient_to_text(row)
    for _, row in val_df.iterrows()
]

test_texts = [
    patient_to_text(row)
    for _, row in test_df.iterrows()
]

print("Training prompts:", len(train_texts))
print("Validation prompts:", len(val_texts))
print("Test prompts:", len(test_texts))

TRAIN_PROMPT_LENGTHS = [
    len(tokenizer.encode(t, add_special_tokens=False))
    for t in train_texts
]
print("Training prompt tokens: min =", min(TRAIN_PROMPT_LENGTHS),
      "| median =", int(np.median(TRAIN_PROMPT_LENGTHS)),
      "| 95th percentile =", int(np.percentile(TRAIN_PROMPT_LENGTHS, 95)),
      "| max =", max(TRAIN_PROMPT_LENGTHS))
MAX_LENGTH = min(
    1024, max(32, int(np.ceil(max(TRAIN_PROMPT_LENGTHS) / 8) * 8))
)
if max(TRAIN_PROMPT_LENGTHS) > 1024:
    print("WARNING: Some training prompts exceed GPT-2's context limit.")
print("Selected MAX_LENGTH:", MAX_LENGTH)

BATCH_SIZE = 64
EPOCHS = 500
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1
print("Batch size:", BATCH_SIZE, "| Epochs:", EPOCHS,
      "| Learning rate:", LEARNING_RATE,
      "| Weight decay:", WEIGHT_DECAY)

# ============================================================
# 8. BUILD DATASETS AND DATA LOADERS
# ============================================================
class HPVTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.items = []
        for text, label in zip(texts, labels):
            encoded = tokenizer(
                text, max_length=MAX_LENGTH, padding="max_length",
                truncation=True, return_tensors="pt"
            )
            self.items.append((
                encoded["input_ids"].squeeze(0),
                encoded["attention_mask"].squeeze(0), int(label)
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        input_ids, attention_mask, label = self.items[index]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(label, dtype=torch.long)
        }

train_dataset = HPVTextDataset(train_texts, y_train)
val_dataset = HPVTextDataset(val_texts, y_val)
test_dataset = HPVTextDataset(test_texts, y_test)
train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=train_generator, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

# ============================================================
# 9. INITIALISE GPT-2 CLASSIFIER AND CLASS WEIGHTS
# ============================================================
c
# 9. INITIALISE GPT-2 CLASSIFIER AND CLASS-WEIGHTED LOSS

class HPVNetGPT2Text(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=2,
                 dropout=DROPOUT):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained(model_name)
        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = tokenizer.pad_token_id
        hidden_size = self.gpt2.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )
        hidden_states = outputs.last_hidden_state
        last_positions = attention_mask.sum(dim=1) - 1
        pooled_output = hidden_states[
            torch.arange(hidden_states.size(0), device=hidden_states.device),
            last_positions
        ]
        return self.classifier(pooled_output)


model = HPVNetGPT2Text().to(device)

# Freeze the GPT-2 backbone, then unfreeze only the final four blocks.
for param in model.gpt2.parameters():
    param.requires_grad = False

for block in model.gpt2.h[-4:]:
    for param in block.parameters():
        param.requires_grad = True

# Keep the classifier head trainable.
for param in model.classifier.parameters():
    param.requires_grad = True

# Verify trainable parameters.
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")
print("Trainable GPT-2 blocks: final 4 only; classifier head: trainable")

# Calculate class weights.
class_counts = np.bincount(y_train, minlength=2)
class_weights = torch.tensor(
    len(y_train) / (2 * np.maximum(class_counts, 1)),
    dtype=torch.float32,
    device=device
)

print(
    "Class weights [negative, positive]:",
    class_weights.detach().cpu().numpy()
)

def weighted_classification_loss(logits, labels):
    losses = nn.functional.cross_entropy(
        logits, labels, reduction="none"
    )
    return (losses * class_weights[labels]).mean()

# Optimise only trainable parameters.
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ============================================================
# 10. HPV SCORES, VALIDATION LOSS AND THRESHOLD SELECTION
# ============================================================
def get_hpv_scores(loader):
    model.eval()
    scores, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask)
            scores.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            labels_all.extend(batch["label"].numpy())
    return np.asarray(scores, dtype=float), np.asarray(labels_all, dtype=int)

def evaluate_loss(loader):
    model.eval()
    total_loss, total_examples = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask)
            loss = weighted_classification_loss(logits, labels)
            n = labels.size(0)
            total_loss += loss.item() * n
            total_examples += n
    return total_loss / max(total_examples, 1)

def select_threshold(scores, labels):
    thresholds = np.unique(np.concatenate(([0.0], scores, [1.0])))
    results = [
        (float(t), balanced_accuracy_score(labels, (scores >= t).astype(int)))
        for t in thresholds
    ]
    # Maximise balanced accuracy; ties go to the threshold closest to 0.5.
    return max(results, key=lambda x: (x[1], -abs(x[0] - 0.5)))

# ============================================================
# 11. TRAIN AND SAVE BEST VALIDATION CHECKPOINTS
# ============================================================
OUT_DIR = Path("ROC_HANCOCK_GPT2_CLASSIFIER")
OUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_paths = {
    "Validation_Loss": OUT_DIR / "best_model_validation_loss.pth",
    "Validation_ROC_AUC": OUT_DIR / "best_model_validation_roc_auc.pth",
    "Validation_PR_AUC": OUT_DIR / "best_model_validation_pr_auc.pth",
    "Validation_Balanced_Accuracy":
        OUT_DIR / "best_model_validation_balanced_accuracy.pth"
}
best = {
    "Validation_Loss": {
        "value": float("inf"), "epoch": None, "threshold": None
    },
    "Validation_ROC_AUC": {
        "value": -float("inf"), "epoch": None, "threshold": None
    },
    "Validation_PR_AUC": {
        "value": -float("inf"), "epoch": None, "threshold": None
    },
    "Validation_Balanced_Accuracy": {
        "value": -float("inf"), "epoch": None, "threshold": None
    }
}
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_examples = 0.0, 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = weighted_classification_loss(logits, labels)
        loss.backward()
        optimizer.step()
        n = labels.size(0)
        train_loss_sum += loss.item() * n
        train_examples += n

    train_loss = train_loss_sum / max(train_examples, 1)
    val_loss = evaluate_loss(val_loader)
    val_scores, val_labels = get_hpv_scores(val_loader)
    val_auc = roc_auc_score(val_labels, val_scores)
    val_ap = average_precision_score(val_labels, val_scores)
    val_threshold, val_ba = select_threshold(val_scores, val_labels)
    history.append({
        "Epoch": epoch, "Train_Loss": train_loss,
        "Validation_Loss": val_loss, "Validation_ROC_AUC": val_auc,
        "Validation_PR_AUC": val_ap,
        "Validation_Balanced_Accuracy": val_ba,
        "Validation_Threshold": val_threshold
    })
    metrics = {
        "Validation_Loss": val_loss, "Validation_ROC_AUC": val_auc,
        "Validation_PR_AUC": val_ap,
        "Validation_Balanced_Accuracy": val_ba
    }
    for metric, value in metrics.items():
        improved = (
            value < best[metric]["value"] if metric == "Validation_Loss"
            else value > best[metric]["value"]
        )
        if improved:
            best[metric] = {
                "value": float(value), "epoch": epoch,
                "threshold": float(val_threshold)
            }
            torch.save(model.state_dict(), checkpoint_paths[metric])

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}/{EPOCHS} | Train Loss={train_loss:.4f} | "
            f"Val Loss={val_loss:.4f} | Val ROC-AUC={val_auc:.4f} | "
            f"Val PR-AUC={val_ap:.4f} | Val BA={val_ba:.4f} | "
            f"Threshold={val_threshold:.6f}"
        )

pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)
best_summary = pd.DataFrame([
    {
        "Checkpoint_Selected_By": metric, "Best_Epoch": info["epoch"],
        "Best_Validation_Metric": info["value"],
        "Threshold_Selected_On_Validation": info["threshold"],
        "Checkpoint_Path": str(checkpoint_paths[metric])
    }
    for metric, info in best.items()
])
best_summary.to_csv(
    OUT_DIR / "best_validation_checkpoints.csv", index=False
)
print("\nBEST VALIDATION CHECKPOINTS\n", best_summary.to_string(index=False))

# ============================================================
# 12. EVALUATE CHECKPOINTS ON THE HELD-OUT TEST SET
# ============================================================
test_results, test_predictions = [], {}
for metric, path in checkpoint_paths.items():
    print(f"\n===== CHECKPOINT: {metric} =====")
    model.load_state_dict(torch.load(path, map_location=device))
    scores, labels = get_hpv_scores(test_loader)
    threshold = best[metric]["threshold"]
    predictions = (scores >= threshold).astype(int)
    auc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)
    ba = balanced_accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, zero_division=0)
    cm = confusion_matrix(labels, predictions, labels=[0, 1])
    print("Best epoch:", best[metric]["epoch"])
    print(f"Validation threshold: {threshold:.6f}")
    print(classification_report(
        labels, predictions, labels=[0, 1],
        target_names=["HPV Negative", "HPV Positive"],
        digits=4, zero_division=0
    ))
    print(
        f"Test ROC-AUC: {auc:.4f} | Test PR-AUC (AP): {ap:.4f} | "
        f"Test Balanced Accuracy: {ba:.4f} | F1: {f1:.4f}"
    )
    print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)
    test_results.append({
        "Dataset": "HANCOCK", "Model": "GPT-2 Classifier",
        "Checkpoint_Selected_By": metric,
        "Best_Epoch": best[metric]["epoch"],
        "Best_Validation_Metric": best[metric]["value"],
        "Threshold_Selected_On_Validation": threshold,
        "Test_ROC_AUC": auc, "Test_PR_AUC_Average_Precision": ap,
        "Test_Balanced_Accuracy": ba, "Test_F1": f1,
        "TN": int(cm[0, 0]), "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]), "TP": int(cm[1, 1])
    })
    test_predictions[metric] = {
        "scores": scores, "predictions": predictions,
        "labels": labels, "threshold": threshold
    }
    pd.DataFrame({
        "True_Label": labels, "HPV_Positive_Score": scores,
        "Predicted_Label": predictions, "Threshold": threshold
    }).to_csv(OUT_DIR / f"test_predictions_{metric.lower()}.csv", index=False)

results_df = pd.DataFrame(test_results)
results_df.to_csv(OUT_DIR / "test_metrics_by_checkpoint.csv", index=False)
print("\nTEST RESULTS BY CHECKPOINT\n", results_df.to_string(index=False))

# ============================================================
# 13. SAVE ROC AND PRECISION-RECALL CURVES FOR ALL CHECKPOINTS
# ============================================================
for metric, result in test_predictions.items():
    labels, scores = result["labels"], result["scores"]
    auc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)
    fpr, tpr, roc_thresholds = roc_curve(labels, scores)
    precision, recall, _ = precision_recall_curve(labels, scores)
    safe_name = metric.lower()
    pd.DataFrame({
        "FPR": fpr, "TPR": tpr, "Threshold": roc_thresholds
    }).to_csv(OUT_DIR / f"roc_coordinates_{safe_name}.csv", index=False)
    pd.DataFrame({
        "Precision": precision, "Recall": recall
    }).to_csv(
        OUT_DIR / f"precision_recall_coordinates_{safe_name}.csv",
        index=False
    )
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f"ROC-AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], "--", linewidth=1.5, label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"HANCOCK GPT-2 Classifier ROC: {metric}")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"roc_curve_{safe_name}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.close()

    plt.figure(figsize=(7, 6))
    plt.plot(recall, precision, linewidth=2, label=f"PR-AUC (AP) = {ap:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"HANCOCK GPT-2 Classifier Precision-Recall: {metric}")
    plt.legend(loc="lower left")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"precision_recall_curve_{safe_name}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.close()

# ============================================================
# 14. SAVE SETTINGS AND APPEND RESULTS TO COMPARISON CSV
# ============================================================
settings = {
    "Dataset": "HANCOCK", "Model": MODEL_NAME,
    "Model_Type": "GPT-2 Classifier",
    "Features": ", ".join(features), "Number_of_Features": len(features),
    "Target": target_col, "Positive_Class": 1,
    "Positive_Class_Name": "HPV Positive",
    "Learning_Rate": LEARNING_RATE, "Weight_Decay": WEIGHT_DECAY,
    "Optimizer": "AdamW", "Epochs": EPOCHS, "Batch_Size": BATCH_SIZE,
    "Max_Length": MAX_LENGTH,
    "Max_Length_Method": (
        "Training prompts only; rounded to multiple of 8; capped at 1024"
    ),
    "Random_Seed": SEED, "Train_Validation_Test_Split": "65/15/20 stratified",
    "SMOTE": False, "Gradient_Clipping": False,
    "GPT2_Backbone_Frozen": False,
    "GPT2_Trainable_Blocks": 4,
    "Dropout": DROPOUT,
    "Class_Weights": str(class_weights.detach().cpu().numpy().tolist()),
    "Checkpoint_Selection": (
        "Best validation loss, ROC-AUC, PR-AUC and balanced accuracy"
    ),
    "Threshold_Selection": (
        "Maximise validation balanced accuracy; ties closest to 0.5"
    ),
    "HPV_Positive_Score": "Softmax probability for class 1 (HPV positive)",
    "Curves": (
        "Test ROC and precision-recall curves saved for each checkpoint"
    )
}
pd.DataFrame([settings]).to_csv(OUT_DIR / "settings.csv", index=False)

comparison_file = Path("model_roc_pr_auc_comparison.csv")
comparison_rows = results_df[[
    "Dataset", "Model", "Checkpoint_Selected_By", "Best_Epoch",
    "Test_ROC_AUC", "Test_PR_AUC_Average_Precision",
    "Test_Balanced_Accuracy", "Test_F1"
]].rename(columns={"Test_PR_AUC_Average_Precision": "Test_PR_AUC"})

if comparison_file.exists():
    existing = pd.read_csv(comparison_file)
    for col in comparison_rows.columns:
        if col not in existing.columns:
            existing[col] = np.nan
    for col in existing.columns:
        if col not in comparison_rows.columns:
            comparison_rows[col] = np.nan
    combined = pd.concat(
        [existing, comparison_rows[existing.columns]], ignore_index=True
    )
    combined = combined.drop_duplicates(
        subset=["Dataset", "Model", "Checkpoint_Selected_By"], keep="last"
    )
else:
    combined = comparison_rows

combined.to_csv(comparison_file, index=False)
print("\nSaved classifier outputs to:", OUT_DIR.resolve())
print("AUC comparison file:", comparison_file.resolve())

Device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 11.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 7.60MB/s]


Clinical shape: (763, 32)
Pathology shape: (763, 18)
Merged shape: (763, 49)
Patients: 332
HPV classes (0=negative, 1=positive):
HPV_Label
0    191
1    141
Name: count, dtype: int64
Features: ['age_at_initial_diagnosis', 'sex', 'smoking_status', 'primary_tumor_site', 'primarily_metastasis', 'pN_stage']
N-stage source: pN_stage
Train: 215 [123  92]
Validation: 50 [29 21]
Test: 67 [39 28]
Number of clinical features: 6
Features: ['age_at_initial_diagnosis', 'sex', 'smoking_status', 'primary_tumor_site', 'primarily_metastasis', 'pN_stage']

Example clinical prompt:
Q: Predict this patient's Human Papillomavirus (HPV) status using the clinical information available. This patient is 60 years old and is male. The patient is a non-smoker. The primary tumour site is Oropharynx. The primarily metastasis status is no. Pathological nodal stage is pN1. Based on these six clinical features, predict whether the patient is HPV positive or HPV negative. Answer with only Yes or No.
A:


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable parameters: 28,451,714 / 124,540,034
Trainable GPT-2 blocks: final 4 only; classifier head: trainable
Class weights [negative, positive]: [0.87398374 1.1684783 ]


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:910.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 50/500 | Train Loss=0.6646 | Val Loss=0.6679 | Val ROC-AUC=0.7069 | Val PR-AUC=0.5898 | Val BA=0.6929 | Threshold=0.646598
Epoch 100/500 | Train Loss=0.6329 | Val Loss=0.6360 | Val ROC-AUC=0.6987 | Val PR-AUC=0.6039 | Val BA=0.7061 | Threshold=0.415692
Epoch 150/500 | Train Loss=0.6393 | Val Loss=0.6737 | Val ROC-AUC=0.5394 | Val PR-AUC=0.4783 | Val BA=0.6486 | Threshold=0.154931
Epoch 200/500 | Train Loss=0.5325 | Val Loss=0.5508 | Val ROC-AUC=0.7923 | Val PR-AUC=0.7939 | Val BA=0.7906 | Threshold=0.342367
Epoch 250/500 | Train Loss=0.5140 | Val Loss=0.6863 | Val ROC-AUC=0.6724 | Val PR-AUC=0.7176 | Val BA=0.7143 | Threshold=0.855551
Epoch 300/500 | Train Loss=0.4611 | Val Loss=0.6065 | Val ROC-AUC=0.6954 | Val PR-AUC=0.7250 | Val BA=0.7036 | Threshold=0.437949
Epoch 350/500 | Train Loss=0.5168 | Val Loss=0.5766 | Val ROC-AUC=0.7956 | Val PR-AUC=0.7698 | Val BA=0.7578 | Threshold=0.526646
Epoch 400/500 | Train Loss=0.5041 | Val Loss=0.6392 | Val ROC-AUC=0.6724 | Val PR-AUC=0.662

In [2]:
# ================================================================
# HANCOCK GPT-2 HPV CLASSIFIER
# SHAP + LIME: ONE ACTUAL POSITIVE AND ONE ACTUAL NEGATIVE PATIENT
# 4 SAME-SIZE PLOTS | 300 DPI | DOWNLOADABLE ZIP
# ================================================================

!pip -q install shap lime

import os
import zipfile
import numpy as np
import pandas as pd
import torch
import shap
import matplotlib.pyplot as plt

from pathlib import Path
from lime.lime_tabular import LimeTabularExplainer
from matplotlib.patches import Patch
from IPython.display import display, FileLink

# ================================================================
# 1. CHECK REQUIRED NOTEBOOK VARIABLES
# ================================================================

required_variables = [
    "model",
    "device",
    "tokenizer",
    "MAX_LENGTH",
    "train_df",
    "test_df",
    "y_test",
    "features",
    "patient_to_text",
    "OUT_DIR"
]

for variable in required_variables:
    if variable not in globals():
        raise NameError(
            f"Missing variable: {variable}. "
            "Run the original GPT-2 notebook first."
        )

print("Required GPT-2 variables verified.")

# ================================================================
# 2. LOAD BEST VALIDATION ROC-AUC CHECKPOINT
# ================================================================

checkpoint_path = (
    Path(OUT_DIR) / "best_model_validation_roc_auc.pth"
)

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {checkpoint_path}"
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()

print("Best validation ROC-AUC checkpoint loaded.")
print("Checkpoint:", checkpoint_path)

# ================================================================
# 3. VERIFY THE SIX CLINICAL FEATURES
# ================================================================

features = [
    "age_at_initial_diagnosis",
    "sex",
    "smoking_status",
    "primary_tumor_site",
    "primarily_metastasis",
    "pN_stage"
]

categorical_features = [
    "sex",
    "smoking_status",
    "primary_tumor_site",
    "primarily_metastasis",
    "pN_stage"
]

assert len(features) == 6

print("Clinical features:", features)

# ================================================================
# 4. PREPARE LABELS AND PATIENT DATA
# ================================================================

y_test_array = np.asarray(
    y_test,
    dtype=int
).ravel()

if len(test_df) != len(y_test_array):
    raise ValueError(
        "test_df and y_test do not have matching lengths."
    )

positive_indices = np.where(
    y_test_array == 1
)[0]

negative_indices = np.where(
    y_test_array == 0
)[0]

if len(positive_indices) == 0:
    raise ValueError(
        "No actual HPV-positive patient in the test set."
    )

if len(negative_indices) == 0:
    raise ValueError(
        "No actual HPV-negative patient in the test set."
    )

positive_idx = int(positive_indices[0])
negative_idx = int(negative_indices[0])

selected_patients = [
    ("HPV_positive", positive_idx, 1),
    ("HPV_negative", negative_idx, 0)
]

print("\nSelected patients:")
print("Positive patient test index:", positive_idx)
print("Negative patient test index:", negative_idx)

# ================================================================
# 5. ENCODE THE SIX CLINICAL FEATURES FOR EXPLANATION
# ================================================================
# GPT-2 receives text, but SHAP and LIME need a feature matrix.
# Categorical values are represented by training-set category codes.
# The prediction function converts the codes back into clinical text.

category_levels = {}
category_to_code = {}

for col in categorical_features:

    levels = sorted(
        train_df[col].astype(str).unique().tolist()
    )

    category_levels[col] = levels

    category_to_code[col] = {
        value: index
        for index, value in enumerate(levels)
    }

    if not levels:
        raise ValueError(
            f"No training categories found for {col}"
        )


def dataframe_to_matrix(dataframe):

    matrix = np.zeros(
        (len(dataframe), len(features)),
        dtype=np.float32
    )

    # Age
    matrix[:, 0] = pd.to_numeric(
        dataframe["age_at_initial_diagnosis"],
        errors="coerce"
    ).fillna(
        pd.to_numeric(
            train_df["age_at_initial_diagnosis"],
            errors="coerce"
        ).median()
    ).to_numpy(dtype=np.float32)

    # Categorical features
    for column_index, col in enumerate(
        categorical_features,
        start=1
    ):

        mapping = category_to_code[col]

        values = dataframe[col].astype(str).tolist()

        matrix[:, column_index] = [
            mapping.get(value, 0)
            for value in values
        ]

    return matrix


X_explain_train = dataframe_to_matrix(train_df)
X_explain_test = dataframe_to_matrix(test_df)

print("\nExplanation training matrix:", X_explain_train.shape)
print("Explanation test matrix:", X_explain_test.shape)

# ================================================================
# 6. CONVERT ENCODED FEATURES BACK INTO CLINICAL TEXT
# ================================================================

def matrix_to_texts(matrix):

    matrix = np.asarray(
        matrix,
        dtype=np.float32
    )

    rows = []

    for values in matrix:

        row = {}

        row["age_at_initial_diagnosis"] = float(
            values[0]
        )

        for column_index, col in enumerate(
            categorical_features,
            start=1
        ):

            levels = category_levels[col]

            code = int(
                np.clip(
                    np.rint(values[column_index]),
                    0,
                    len(levels) - 1
                )
            )

            row[col] = levels[code]

        rows.append(
            patient_to_text(pd.Series(row))
        )

    return rows


# ================================================================
# 7. GPT-2 PREDICTION FUNCTION FOR SHAP AND LIME
# ================================================================

def gpt2_predict_proba(data):

    data = np.asarray(
        data,
        dtype=np.float32
    )

    if data.ndim == 1:
        data = data.reshape(1, -1)

    if data.shape[1] != len(features):
        raise ValueError(
            f"Expected {len(features)} features, "
            f"received {data.shape[1]}"
        )

    texts = matrix_to_texts(data)

    model.eval()

    all_probabilities = []

    # Small batches help avoid GPU memory problems
    batch_size = 8

    with torch.no_grad():

        for start in range(0, len(texts), batch_size):

            batch_texts = texts[
                start:start + batch_size
            ]

            encoded = tokenizer(
                batch_texts,
                max_length=MAX_LENGTH,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

            input_ids = encoded[
                "input_ids"
            ].to(device)

            attention_mask = encoded[
                "attention_mask"
            ].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            all_probabilities.append(
                probabilities.cpu().numpy()
            )

    return np.vstack(all_probabilities)


# Test GPT-2 prediction function
prediction_test = gpt2_predict_proba(
    X_explain_test[:3]
)

print("\nPrediction test shape:", prediction_test.shape)

if prediction_test.shape != (3, 2):
    raise ValueError("GPT-2 prediction test failed.")

print("GPT-2 prediction function working.")

# ================================================================
# 8. CREATE OUTPUT FOLDER
# ================================================================

OUTPUT_DIR = (
    Path(OUT_DIR) / "GPT2_SHAP_LIME_Individual_Plots"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nOutput folder:", OUTPUT_DIR)

# ================================================================
# 9. CREATE SHAP BACKGROUND DATA
# ================================================================

np.random.seed(42)

background_size = min(
    15,
    len(X_explain_train)
)

background_indices = np.random.choice(
    len(X_explain_train),
    size=background_size,
    replace=False
)

background = X_explain_train[
    background_indices
]

# ================================================================
# 10. CREATE LIME EXPLAINER
# ================================================================

lime_categorical_indices = [
    features.index(col)
    for col in categorical_features
]

lime_categorical_names = {
    features.index(col): category_levels[col]
    for col in categorical_features
}

lime_explainer = LimeTabularExplainer(
    training_data=X_explain_train,
    feature_names=features,
    class_names=[
        "HPV-negative",
        "HPV-positive"
    ],
    categorical_features=lime_categorical_indices,
    categorical_names=lime_categorical_names,
    mode="classification",
    discretize_continuous=False,
    random_state=42
)

# ================================================================
# 11. HELPER FOR ORIGINAL PATIENT VALUES
# ================================================================

def get_patient_display_values(patient_idx):

    row = test_df.iloc[patient_idx]

    values = []

    for feature in features:

        value = row[feature]

        if pd.isna(value):
            value = "Unknown"

        elif isinstance(value, (float, np.floating)):
            if float(value).is_integer():
                value = str(int(value))
            else:
                value = str(value)

        else:
            value = str(value)

        values.append(value)

    return np.array(values, dtype=object)


# ================================================================
# 12. GENERATE SHAP AND LIME FOR TWO PATIENTS
# ================================================================

for patient_name, patient_idx, class_label in selected_patients:

    print("\n" + "=" * 65)
    print("EXPLAINING:", patient_name)
    print("=" * 65)

    patient = X_explain_test[
        patient_idx
    ]

    original_values = get_patient_display_values(
        patient_idx
    )

    # ------------------------------------------------------------
    # DISPLAY ACTUAL CLASS AND MODEL PROBABILITIES
    # ------------------------------------------------------------

    probabilities = gpt2_predict_proba(
        patient.reshape(1, -1)
    )[0]

    print("Actual HPV label:", class_label)

    print(
        "HPV-negative probability:",
        round(float(probabilities[0]), 4)
    )

    print(
        "HPV-positive probability:",
        round(float(probabilities[1]), 4)
    )

    # ------------------------------------------------------------
    # A. SHAP
    # ------------------------------------------------------------

    print("\nCalculating SHAP values...")

    def predict_selected_class(data):
        return gpt2_predict_proba(data)[:, class_label]

    shap_explainer = shap.KernelExplainer(
        predict_selected_class,
        background
    )

    shap_values = shap_explainer.shap_values(
        patient.reshape(1, -1),
        nsamples=200
    )

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(
        shap_values
    ).reshape(-1)

    base_value = float(
        np.asarray(
            shap_explainer.expected_value
        ).reshape(-1)[0]
    )

    shap_explanation = shap.Explanation(
        values=shap_values,
        base_values=base_value,
        data=original_values,
        feature_names=features
    )

    # Fixed size for every SHAP plot
    plt.figure(figsize=(8, 4.8))

    shap.plots.waterfall(
        shap_explanation,
        max_display=6,
        show=False
    )

    fig = plt.gcf()
    fig.set_size_inches(8, 4.8)

    plt.title(
        f"SHAP Explanation - {patient_name}",
        fontsize=12
    )

    plt.tight_layout()

    shap_filename = (
        OUTPUT_DIR /
        f"{patient_name}_SHAP_waterfall.png"
    )

    plt.savefig(
        shap_filename,
        dpi=300,
        facecolor="white"
    )

    plt.close("all")

    print("SHAP saved:", shap_filename)

    # ------------------------------------------------------------
    # B. LIME
    # ------------------------------------------------------------

    print("\nCalculating LIME values...")

    lime_explanation = lime_explainer.explain_instance(
        data_row=patient,
        predict_fn=gpt2_predict_proba,
        labels=(class_label,),
        num_features=6,
        num_samples=1000
    )

    lime_map = lime_explanation.as_map()[
        class_label
    ]

    lime_values = np.zeros(
        len(features),
        dtype=float
    )

    for feature_index, weight in lime_map:

        lime_values[feature_index] = float(weight)

    # ------------------------------------------------------------
    # C. LIME BAR CHART
    # ------------------------------------------------------------

    order = np.argsort(
        np.abs(lime_values)
    )

    ordered_features = np.array(
        features
    )[order]

    ordered_values = lime_values[
        order
    ]

    # Red = toward the selected class
    # Green = away from the selected class

    colors = [
        "red" if value >= 0 else "green"
        for value in ordered_values
    ]

    fig, ax = plt.subplots(
        figsize=(8, 4.8)
    )

    ax.barh(
        ordered_features,
        ordered_values,
        color=colors,
        edgecolor="black",
        linewidth=0.4
    )

    ax.axvline(
        0,
        color="black",
        linewidth=0.8
    )

    ax.set_xlabel(
        "LIME contribution toward selected class",
        fontsize=10
    )

    ax.set_ylabel(
        "Clinical features",
        fontsize=10
    )

    ax.set_title(
        f"LIME Explanation - {patient_name}",
        fontsize=12
    )

    ax.legend(
        handles=[
            Patch(
                color="red",
                label="Positive contribution"
            ),
            Patch(
                color="green",
                label="Negative contribution"
            )
        ],
        loc="best",
        fontsize=8
    )

    ax.tick_params(
        axis="both",
        labelsize=9
    )

    plt.tight_layout()

    lime_filename = (
        OUTPUT_DIR /
        f"{patient_name}_LIME_bar.png"
    )

    plt.savefig(
        lime_filename,
        dpi=300,
        facecolor="white"
    )

    plt.close("all")

    print("LIME saved:", lime_filename)


# ================================================================
# 13. VERIFY ALL FOUR PLOTS
# ================================================================

expected_files = [
    "HPV_positive_SHAP_waterfall.png",
    "HPV_positive_LIME_bar.png",
    "HPV_negative_SHAP_waterfall.png",
    "HPV_negative_LIME_bar.png"
]

print("\n" + "=" * 65)
print("VERIFYING GENERATED PLOTS")
print("=" * 65)

for filename in expected_files:

    filepath = OUTPUT_DIR / filename

    if not filepath.exists():
        raise FileNotFoundError(
            f"Missing plot: {filename}"
        )

    image = plt.imread(filepath)

    print(
        "FOUND:",
        filename,
        "| Dimensions:",
        image.shape
    )

# ================================================================
# 14. CREATE ZIP FILE
# ================================================================

zip_path = (
    OUTPUT_DIR /
    "ALL_4_GPT2_HPV_SHAP_LIME_PLOTS.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for filename in expected_files:

        filepath = OUTPUT_DIR / filename

        zip_file.write(
            filepath,
            arcname=filename
        )

print("\nZIP created successfully:", zip_path)

# ================================================================
# 15. DISPLAY FOUR INDIVIDUAL DOWNLOAD LINKS
# ================================================================

print("\nINDIVIDUAL PLOT DOWNLOADS:")

for filename in expected_files:

    display(
        FileLink(
            str(OUTPUT_DIR / filename)
        )
    )

# ================================================================
# 16. DISPLAY ZIP DOWNLOAD LINK
# ================================================================

print("\nDOWNLOAD ALL FOUR PLOTS TOGETHER:")

display(
    FileLink(str(zip_path))
)

# ================================================================
# 17. AUTOMATIC GOOGLE COLAB DOWNLOAD
# ================================================================

try:

    from google.colab import files

    files.download(str(zip_path))

    print("\nAutomatic ZIP download initiated.")

except ImportError:

    print(
        "Use the ZIP download link above."
    )

print("\nGPT-2 SHAP AND LIME GENERATION COMPLETED.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Required GPT-2 variables verified.
Best validation ROC-AUC checkpoint loaded.
Checkpoint: ROC_HANCOCK_GPT2_CLASSIFIER/best_model_validation_roc_auc.pth
Clinical features: ['age_at_initial_diagnosis', 'sex', 'smoking_status', 'primary_tumor_site', 'primarily_metastasis', 'pN_stage']

Selected patients:
Positive patient test index: 0
Negative patient test index: 2

Explanation training matrix: (215, 6)
Explanation test matrix: (67, 6)

Prediction test shape: (3, 2)
GPT-2 prediction function working.

Output folder: ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots

EXPLAINING: HPV_positive
Actual HPV label: 1
HPV-negative probability: 0.2324
HPV-positive probability: 0.7676

Calculating SHAP values...


  0%|          | 0/1 [00:00<?, ?it/s]

SHAP saved: ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots/HPV_positive_SHAP_waterfall.png

Calculating LIME values...
LIME saved: ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots/HPV_positive_LIME_bar.png

EXPLAINING: HPV_negative
Actual HPV label: 0
HPV-negative probability: 0.7937
HPV-positive probability: 0.2063

Calculating SHAP values...


  0%|          | 0/1 [00:00<?, ?it/s]

SHAP saved: ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots/HPV_negative_SHAP_waterfall.png

Calculating LIME values...
LIME saved: ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots/HPV_negative_LIME_bar.png

VERIFYING GENERATED PLOTS
FOUND: HPV_positive_SHAP_waterfall.png | Dimensions: (1440, 2400, 4)
FOUND: HPV_positive_LIME_bar.png | Dimensions: (1440, 2400, 4)
FOUND: HPV_negative_SHAP_waterfall.png | Dimensions: (1440, 2400, 4)
FOUND: HPV_negative_LIME_bar.png | Dimensions: (1440, 2400, 4)

ZIP created successfully: ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots/ALL_4_GPT2_HPV_SHAP_LIME_PLOTS.zip

INDIVIDUAL PLOT DOWNLOADS:


/content/ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots/HPV_positive_SHAP_waterfall.png

... (4 outputs left out)

/content/ROC_HANCOCK_GPT2_CLASSIFIER/GPT2_SHAP_LIME_Individual_Plots/ALL_4_GPT2_HPV_SHAP_LIME_PLOTS.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Automatic ZIP download initiated.

GPT-2 SHAP AND LIME GENERATION COMPLETED.
